In [ ]:
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import spikeinterface.extractors as se

possible_roots = [
    Path.cwd() / "data",
    Path.cwd().parent / "data",
    Path("data"),
    Path("../data"),
]

data_root = next((root.resolve() for root in possible_roots if root.exists()), None)
if data_root is None:
    raise FileNotFoundError("Could not find the data folder. Expected a 'data' directory near the notebook.")

stream_id = "0"
seconds_per_file = 2.0

folder_to_files = defaultdict(list)
for file_path in sorted(data_root.rglob("*.rhd")):
    folder_to_files[file_path.parent].append(file_path)

if not folder_to_files:
    print(f"No .rhd files found under {data_root}")
else:
    for folder in sorted(folder_to_files):
        files = folder_to_files[folder]
        fig_height = max(2.5, 2.25 * len(files))
        fig, axes = plt.subplots(
            len(files),
            1,
            figsize=(16, fig_height),
            sharex=True,
            constrained_layout=True,
        )

        if len(files) == 1:
            axes = [axes]

        for ax, file_path in zip(axes, files):
            try:
                recording = se.read_intan(str(file_path), stream_id=stream_id)
                sampling_hz = float(recording.get_sampling_frequency())
                n_samples = int(recording.get_num_samples())
                n_channels = int(recording.get_num_channels())
                channel_id = recording.get_channel_ids()[0]
                frame_count = max(1, min(n_samples, int(sampling_hz * seconds_per_file)))
                traces = recording.get_traces(start_frame=0, end_frame=frame_count, channel_ids=[channel_id])
                signal = traces[:, 0]
                time_axis = np.arange(signal.shape[0]) / sampling_hz

                ax.plot(time_axis, signal, linewidth=0.8, color="tab:blue")
                ax.set_title(
                    f"{file_path.name} | {n_channels} ch | {n_samples / sampling_hz:.2f} s",
                    loc="left",
                    fontsize=10,
                )
                ax.set_ylabel("amplitude")
                ax.grid(True, alpha=0.25)
            except Exception as exc:
                ax.text(
                    0.5,
                    0.5,
                    f"Failed to load:\n{file_path.name}\n{exc}",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                )
                ax.set_axis_off()

        axes[-1].set_xlabel("time (s)")
        folder_label = folder.relative_to(data_root)
        fig.suptitle(str(folder_label) if str(folder_label) != "." else data_root.name, fontsize=14)
        plt.show()


In [ ]:
# knowledge of mat files
from scipy.io import loadmat
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# knowledge of mat files
mat_root = data_root
mat_files = sorted(mat_root.rglob("*.mat"))

if not mat_files:
    print(f"No .mat files found under {mat_root}")
else:
    def describe_value(value, name="value", indent=0):
        prefix = "  " * indent
        if isinstance(value, np.ndarray):
            print(f"{prefix}{name}: ndarray shape={value.shape}, dtype={value.dtype}")
            if value.dtype.names:
                print(f"{prefix}  fields={value.dtype.names}")
            return
        if isinstance(value, np.void):
            field_names = value.dtype.names or ()
            print(f"{prefix}{name}: struct with fields={field_names}")
            for field_name in field_names:
                describe_value(value[field_name], field_name, indent + 1)
            return
        if isinstance(value, dict):
            print(f"{prefix}{name}: dict with keys={list(value.keys())}")
            for key_name, item in value.items():
                describe_value(item, key_name, indent + 1)
            return
        print(f"{prefix}{name}: {type(value).__name__} -> {value}")

    for file_path in mat_files:
        print("\n" + "=" * 100)
        print(f"FILE: {file_path.relative_to(mat_root)}")
        print("=" * 100)

        mat_data = loadmat(str(file_path), squeeze_me=False, struct_as_record=False)
        data_keys = [key for key in mat_data.keys() if not key.startswith("__")]

        if not data_keys:
            print("No non-metadata variables found in this file.")
            continue

        print(f"Variables: {data_keys}")

        for key in data_keys:
            describe_value(mat_data[key], key)

        for key in data_keys:
            value = mat_data[key]
            if not isinstance(value, np.ndarray):
                continue
            if value.size == 0:
                continue
            if not np.issubdtype(value.dtype, np.number):
                continue

            arr = np.asarray(value)
            print(f"\nPlotting variable '{key}' with shape {arr.shape}")

            if arr.ndim == 1:
                fig, ax = plt.subplots(figsize=(14, 3))
                ax.plot(arr, linewidth=0.8)
                ax.set_title(f"{file_path.name} | {key} | shape={arr.shape}")
                ax.set_xlabel("sample index")
                ax.set_ylabel("value")
                ax.grid(True, alpha=0.25)
                plt.show()
            elif arr.ndim == 2:
                fig, ax = plt.subplots(figsize=(14, 3))
                if arr.shape[1] == 2:
                    ax.plot(arr[:, 0], arr[:, 1], linewidth=0.8)
                    ax.set_xlabel("column 0")
                    ax.set_ylabel("column 1")
                    ax.set_title(f"{file_path.name} | {key} | shape={arr.shape} | 2-column view")
                else:
                    if arr.shape[0] >= arr.shape[1]:
                        y = arr[:, 0]
                    else:
                        y = arr[0, :]
                    ax.plot(y, linewidth=0.8)
                    ax.set_xlabel("sample index")
                    ax.set_ylabel("value")
                    ax.set_title(f"{file_path.name} | {key} | shape={arr.shape} | first vector")
                ax.grid(True, alpha=0.25)
                plt.show()
            else:
                print(f"Skipping plot for '{key}' because it has ndim={arr.ndim} and shape={arr.shape}")

In [ ]:
from scipy.io import loadmat
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

mat_root = data_root
mat_files = sorted(mat_root.rglob("*.mat"))

if not mat_files:
    print(f"No .mat files found under {mat_root}")
else:
    print("Found .mat files:")
    for i, file_path in enumerate(mat_files, start=1):
        print(f"{i}. {file_path.relative_to(mat_root)}")

    # Change this to inspect a different file
    selected_index = 0
    file_path = mat_files[selected_index]

    print("\n" + "=" * 100)
    print(f"INSPECTING: {file_path.relative_to(mat_root)}")
    print("=" * 100)

    mat_data = loadmat(str(file_path), squeeze_me=False, struct_as_record=False)
    data_keys = [key for key in mat_data.keys() if not key.startswith("__")]

    if not data_keys:
        print("No non-metadata variables found in this file.")
    else:
        print(f"Variables: {data_keys}")

        for key in data_keys:
            value = mat_data[key]
            if isinstance(value, np.ndarray):
                print(f"{key}: shape={value.shape}, dtype={value.dtype}")
            else:
                print(f"{key}: type={type(value).__name__}")

        key = data_keys[0]
        data = mat_data[key]

        if not isinstance(data, np.ndarray):
            print(f"\nVariable '{key}' is not an ndarray, so it cannot be plotted directly.")
        elif data.size == 0:
            print(f"\nVariable '{key}' is empty.")
        elif not np.issubdtype(data.dtype, np.number):
            print(f"\nVariable '{key}' is not numeric.")
        else:
            arr = np.asarray(data)

            print(f"\nSelected variable: {key}")
            print(f"Shape: {arr.shape}")
            print(f"Dtype: {arr.dtype}")

            if arr.ndim == 2 and arr.shape[1] == 2:
                col0 = arr[:, 0]
                col1 = arr[:, 1]

                print("\nColumn 0 stats:")
                print(f"  min={np.min(col0):.6g}, max={np.max(col0):.6g}, mean={np.mean(col0):.6g}, std={np.std(col0):.6g}")
                print("Column 1 stats:")
                print(f"  min={np.min(col1):.6g}, max={np.max(col1):.6g}, mean={np.mean(col1):.6g}, std={np.std(col1):.6g}")

                is_time_like = np.all(np.diff(col0) >= 0)
                print(f"\nColumn 0 looks time-like: {is_time_like}")
                if is_time_like:
                    print("Interpretation: column 0 is likely time, column 1 is likely angle/signal.")

                fig, axes = plt.subplots(3, 1, figsize=(14, 10), constrained_layout=True)

                axes[0].plot(col0, linewidth=0.9, color="tab:blue")
                axes[0].set_title(f"{file_path.name} | {key} | column 0")
                axes[0].set_ylabel("col 0")
                axes[0].grid(True, alpha=0.25)

                axes[1].plot(col1, linewidth=0.9, color="tab:green")
                axes[1].set_title(f"{file_path.name} | {key} | column 1")
                axes[1].set_ylabel("col 1")
                axes[1].grid(True, alpha=0.25)

                axes[2].plot(col0, col1, linewidth=0.9, color="tab:purple")
                axes[2].set_title(f"{file_path.name} | {key} | col 0 vs col 1")
                axes[2].set_xlabel("col 0")
                axes[2].set_ylabel("col 1")
                axes[2].grid(True, alpha=0.25)

                plt.show()
            elif arr.ndim == 1:
                print("\n1D array stats:")
                print(f"  min={np.min(arr):.6g}, max={np.max(arr):.6g}, mean={np.mean(arr):.6g}, std={np.std(arr):.6g}")

                fig, ax = plt.subplots(figsize=(14, 3))
                ax.plot(arr, linewidth=0.9, color="tab:blue")
                ax.set_title(f"{file_path.name} | {key} | shape={arr.shape}")
                ax.set_xlabel("sample index")
                ax.set_ylabel("value")
                ax.grid(True, alpha=0.25)
                plt.show()
            else:
                print(f"\nSkipping plot for '{key}' because it has ndim={arr.ndim} and shape={arr.shape}")

from above we gather:
that **column 0 is in seconds.**

Here is the breakdown of the specific evidence from the study ("Classification of naturally evoked compound action potentials in peripheral nerve spatiotemporal recordings") and the data in your image:

### 1. The Frame Rate Calculation
In the **Methods** section of the paper, the authors state that the ankle rotation (the mechanical stimulus) was recorded using a standard Logitech webcam at **30 frames per second (fps)**.

* **Look at your X-axis:** In the top two plots, the horizontal axis goes from 0 to roughly **10,500**. These represent the individual video frames.
* **Look at the Y-axis of the top plot:** At the very end of the recording (frame 10,500), the value of `col 0` is exactly **350**.
* **The Math:** $10,500 \text{ frames} \div 30 \text{ frames per second} = 350 \text{ seconds}$.

This linear relationship ($y = \frac{1}{30}x$) confirms that column 0 is the conversion of the frame count into total elapsed time in seconds.



### 2. The Nature of the Experiment
The experiment involved manually rotating a rat's ankle to evoke neural signals. 
* Doing this for **350 seconds** (about 5 minutes and 50 seconds) is a standard duration for a laboratory recording session to gather enough "trials" for machine learning.
* If the unit were **minutes**, the experiment would have lasted nearly 6 hours ($350 \text{ minutes}$), which is unlikely for this type of acute physiological preparation.
* If the unit were **milliseconds**, the entire recording would have lasted only 0.35 seconds, which is impossible given that the middle plot shows dozens of slow, manual oscillations of the ankle joint.

### 3. The Relationship in the Bottom Plot
The bottom plot is a "Phase Plot" or a time-series plot where `col 1` (Angle) is plotted against `col 0` (Time). 
* The x-axis here goes from **0 to 350**. 
* Because we know the middle plot shows the ankle moving back and forth (60° to -60°), and it takes about 5-8 seconds per full oscillation (which is a natural human hand-movement speed), the 350-second duration fits the physical reality of the experiment perfectly.

**Summary:**
* **col 0:** Time in **seconds**.
* **col 1:** Ankle angle in **degrees**.
* **X-axis (top/mid):** Raw **frame number** from the 30fps video.

In [ ]:
from scipy.io import loadmat
from collections import defaultdict
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Find all .mat files in data directory
mat_root = data_root  # Reuse the data_root from previous cell
mat_folder_to_files = defaultdict(list)

for file_path in sorted(mat_root.rglob("*.mat")):
    mat_folder_to_files[file_path.parent].append(file_path)

if not mat_folder_to_files:
    print(f"No .mat files found under {mat_root}")
else:
    for folder in sorted(mat_folder_to_files):
        files = mat_folder_to_files[folder]
        fig_height = max(2.5, 2.25 * len(files))
        fig, axes = plt.subplots(
            len(files),
            1,
            figsize=(16, fig_height),
            sharex=False,
            constrained_layout=True,
        )

        if len(files) == 1:
            axes = [axes]

        for ax, file_path in zip(axes, files):
            try:
                mat_data = loadmat(str(file_path))
                
                # Filter out MATLAB metadata keys
                data_keys = [k for k in mat_data.keys() if not k.startswith('__')]
                
                if not data_keys:
                    ax.text(0.5, 0.5, f"No data arrays in:\n{file_path.name}", 
                           ha="center", va="center", transform=ax.transAxes)
                    ax.set_axis_off()
                    continue
                
                # Use the first data array found
                key = data_keys[0]
                data = mat_data[key]
                
                # Handle 2D arrays (matrix data)
                if data.ndim == 2 and data.shape[1] == 2:
                    # Plot col 0 vs col 1
                    col0 = data[:, 0]
                    col1 = data[:, 1]
                    ax.plot(col0, col1, linewidth=0.8, color="tab:purple")
                    ax.set_xlabel("col 0 (time in seconds)")
                    ax.set_ylabel("col 1 (angle in degrees)")
                elif data.ndim == 2:
                    if data.shape[0] > data.shape[1]:
                        signal = data[:, 0]
                    else:
                        signal = data[0, :]
                    ax.plot(signal, linewidth=0.8, color="tab:green")
                    ax.set_ylabel("amplitude")
                elif data.ndim == 1:
                    ax.plot(data, linewidth=0.8, color="tab:blue")
                    ax.set_ylabel("value")
                else:
                    ax.text(0.5, 0.5, f"Unsupported data shape:\n{data.shape}", 
                           ha="center", va="center", transform=ax.transAxes)
                    ax.set_axis_off()
                    continue
                
                ax.set_title(
                    f"{file_path.name} | shape={data.shape} | key='{key}'",
                    loc="left",
                    fontsize=10,
                )
                ax.grid(True, alpha=0.25)
                
            except Exception as exc:
                ax.text(
                    0.5,
                    0.5,
                    f"Failed to load:\n{file_path.name}\n{type(exc).__name__}: {str(exc)[:50]}",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                    fontsize=9,
                )
                ax.set_axis_off()

        folder_label = folder.relative_to(mat_root)
        fig.suptitle(f".mat files: {str(folder_label) if str(folder_label) != '.' else mat_root.name}", fontsize=14)
        plt.show()


In [ ]:
from scipy.io import loadmat
import numpy as np
import matplotlib.pyplot as plt
import re
from collections import defaultdict

# 1. Group folders by their base name (removing _PART1, _PART2, etc.)
groups = defaultdict(lambda: {"rhd": [], "mat": []})

# Iterate through all identified folders (from your folder_to_files and mat_folder_to_files)
all_folders = set(folder_to_files.keys()).union(set(mat_folder_to_files.keys()))

for folder in all_folders:
    # Use regex to find the base name (e.g., "RAT10_DORSI_PLANTAR")
    base_name = re.sub(r'_PART\d+$', '', folder.name)
    
    if folder in folder_to_files:
        groups[base_name]["rhd"].extend(folder_to_files[folder])
    if folder in mat_folder_to_files:
        groups[base_name]["mat"].extend(mat_folder_to_files[folder])

# 2. Process each combined group
for trial_name, files in groups.items():
    if not files["rhd"] or not files["mat"]:
        print(f"Skipping {trial_name}: Missing either .rhd or .mat files across parts.")
        continue

    # Sort ALL rhd files by name to ensure chronological order across PART1 and PART2
    rhd_files = sorted(files["rhd"], key=lambda f: f.name)
    mat_files = sorted(files["mat"]) # Usually just one .mat file

    all_rhd_signals = []
    all_rhd_times = []
    current_time_offset = 0.0
    sampling_hz = None

    print(f"Processing Trial: {trial_name} ({len(rhd_files)} total files across parts)")

    # 3. Concatenate all .rhd files
    for rhd_file in rhd_files:
        try:
            recording = se.read_intan(str(rhd_file), stream_id=stream_id)
            if sampling_hz is None:
                sampling_hz = float(recording.get_sampling_frequency())
            
            n_samples = int(recording.get_num_samples())
            channel_id = recording.get_channel_ids()[0]
            
            traces = recording.get_traces(start_frame=0, end_frame=n_samples, channel_ids=[channel_id])
            rhd_signal = traces[:, 0]
            
            rhd_time = np.arange(n_samples) / sampling_hz + current_time_offset
            
            all_rhd_signals.append(rhd_signal)
            all_rhd_times.append(rhd_time)
            
            current_time_offset += (n_samples / sampling_hz)
            
        except Exception as e:
            print(f"Error loading {rhd_file.name}: {e}")

    if not all_rhd_signals:
        continue

    full_rhd_signal = np.concatenate(all_rhd_signals)
    full_rhd_time = np.concatenate(all_rhd_times)

    # 4. Load .mat file and plot
    try:
        mat_data = loadmat(str(mat_files[0]))
        data_keys = [k for k in mat_data.keys() if not k.startswith('__')]
        mat_array = np.asarray(mat_data[data_keys[0]])

        if mat_array.ndim >= 2 and mat_array.shape[1] >= 2:
            mat_time = np.asarray(mat_array[:, 0], dtype=float)
            mat_angle = np.asarray(mat_array[:, 1], dtype=float)

            # Align concatenated signal to MAT time
            rhd_signal_on_mat = np.interp(mat_time, full_rhd_time, full_rhd_signal, left=np.nan, right=np.nan)

            fig, ax = plt.subplots(figsize=(14, 6))
            ax2 = ax.twinx()

            ax.plot(mat_time, rhd_signal_on_mat, linewidth=0.5, color="tab:blue", label="Combined RHD (All Parts)", alpha=0.8)
            ax2.plot(mat_time, mat_angle, linewidth=1.0, color="tab:orange", label=".mat angle")

            ax.set_xlabel("Time (s)")
            ax.set_ylabel("RHD Amplitude", color="tab:blue")
            ax2.set_ylabel("Angle (deg)", color="tab:orange")
            ax.set_title(f"Trial: {trial_name}\nTotal Duration: {current_time_offset:.2f}s")
            ax.grid(True, alpha=0.3)
            
            plt.show()
    except Exception as e:
        print(f"Error processing .mat for {trial_name}: {e}")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import scipy.io

# Bemærk: Du skal bruge en Python RHD-læser. 
# En populær løsning er 'intan_io' eller 'pyintan'.
# Hvis du ikke har en, kan man bruge en 'rhd_reader.py' port af Intans kode.

def get_metadata_from_path(file_path):
    """
    Analyserer stien fra manifestet for at finde Rat ID og Target Variable.
    """
    parts = file_path.split('/')
    folder = parts[0] # F.eks. 'RAT10_DORSIFLEXION'
    filename = parts[-1]
    
    # Udtræk Rat ID
    rat_id = folder.split('_')[0]
    
    # Definer Target Variable baseret på mappenavn fra manifestet [source: 4]
    target = "Unknown"
    if "DORSIFLEXION" in folder:
        target = "Dorsiflexion"
    elif "PLANTARFLEXION" in folder:
        target = "Plantarflexion"
    elif "PRICKING" in folder:
        target = "Pricking"
    elif "DORSI_PLANTAR" in folder:
        target = "Dynamic (Dorsi-Plantar)"
        
    return rat_id, target

def clean_rat7_data(data, rat_id):
    """Fjerner kanaler 57-64 hvis det er RAT7."""
    if rat_id == "RAT7" and data.shape[0] >= 64:
        # NumPy slice er 0-indekseret, så 57-64 er index 56 til 64
        data = np.delete(data, slice(56, 64), axis=0)
    return data

In [ ]:
# Eksempel på filer fra dit manifest [source: 4]
manifest_files = [
    "RAT10_DORSIFLEXION/dorsi_170605_122607.rhd",
    "RAT10_PLANTARFLEXION/plantar_170605_123135.rhd",
    "RAT10_PRICKING/prick_170605_124216.rhd",
    "RAT7_DORSI_PLANTAR_PART2/dorsiplantar_170508_115954.rhd"
]

def process_and_plot(file_list, base_path="."):
    for rel_path in file_list:
        full_path = os.path.join(base_path, rel_path)
        rat_id, target = get_metadata_from_path(rel_path)
        
        print(f"Behandler: {rat_id} | Type: {target}")
        
        # Her ville du indlæse din data:
        # data = load_rhd(full_path) 
        # signal = data['amplifier_data']
        
        # Simuleret signal til illustration af plot-strukturen
        fs = 30000 # 30 kHz
        time = np.linspace(0, 1, fs) 
        dummy_signal = np.random.normal(0, 1, fs)

        # Visualisering
        plt.figure(figsize=(15, 5))
        plt.plot(time, dummy_signal, color='blue', linewidth=0.5)
        
        plt.title(f"Rat: {rat_id} | Target: {target} | Fil: {os.path.basename(rel_path)}")
        plt.xlabel("Tid (sekunder)")
        plt.ylabel("Spænding (uV)")
        plt.grid(True, alpha=0.3)
        
        # Særlige noter baseret på dit projekt
        if target == "Pricking":
            plt.annotate('Manuel stimulering foregår her', xy=(0.5, 0), color='red')
        elif "Dynamic" in target:
            plt.annotate('Kræver synkronisering med .mat fil', xy=(0.1, 0), color='green')
            
        plt.tight_layout()
        plt.show()

# Kør funktionen
process_and_plot(manifest_files)

In [ ]:
import shutil, os
shutil.rmtree('__pycache__', ignore_errors=True)
print('Cache slettet – genstart kernel nu')

In [ ]:
import importlib, utils
importlib.reload(utils)
from utils import read_rhd, summarize

data = read_rhd("../data/RAT3_PRICKING/pricking_170227_133227.rhd")
summarize(data)

In [ ]:
import importlib, utils
importlib.reload(utils)
from utils import read_rhd, summarize

data = read_rhd("../data/RAT7_DORSIFLEXION/dorsi_170508_114443.rhd")
summarize(data)

In [ ]:
import importlib, utils
importlib.reload(utils)
from utils import read_rhd
import numpy as np
import matplotlib.pyplot as plt

# Load your file (adjust path if needed)
filepath = "../data/RAT7_DORSIFLEXION/dorsi_170508_114443.rhd"
data = read_rhd(filepath)

amp = data.get('amplifier_data')
if amp is None:
    raise RuntimeError("No 'amplifier_data' in the file")

# Determine sample length and sampling rate
amp_samples = amp.shape[1] if amp.ndim == 2 else amp.shape[0]
fs = data.get('frequency_parameters', {}).get('amplifier_sample_rate', None)
t = np.arange(amp_samples) / fs if fs else np.arange(amp_samples)

candidates = []
excluded = []
# Collect arrays that have any axis equal to amp_samples (exclude non-ndarray and empty)
for key, val in data.items():
    if not isinstance(val, np.ndarray):
        excluded.append((key, getattr(val, '__class__', type(val)).__name__, 'not ndarray'))
        continue
    if val.size == 0:
        excluded.append((key, val.shape, 'empty'))
        continue
    # Exclude artificial keys that reference .mat files (not part of RHD dicts)
    if isinstance(key, str) and key.lower().endswith('.mat'):
        excluded.append((key, getattr(val, 'shape', None), 'mat source - skipped'))
        continue
    # Accept if any axis equals amp_samples
    matching_axes = [i for i, s in enumerate(val.shape) if s == amp_samples]
    if not matching_axes:
        excluded.append((key, val.shape, 'no matching axis length'))
        continue
    match_axis = matching_axes[0]
    # Reduce to 1D by averaging over other axes (keep the axis with amp_samples)
    if val.ndim == 1:
        series = val.astype(float)
    else:
        other_axes = tuple(i for i in range(val.ndim) if i != match_axis)
        if other_axes:
            try:
                series = np.mean(val, axis=other_axes).astype(float)
            except Exception as e:
                excluded.append((key, val.shape, f'reduction error: {e}'))
                continue
        else:
            series = val.astype(float)
        series = np.ravel(series)
    if series.size != amp_samples:
        excluded.append((key, val.shape, 'reduction produced different length'))
        continue
    candidates.append((key, series, val.shape))

# Print detailed report
print("Included arrays (key -> reduced_shape, original_shape):")
for k, s, shp in candidates:
    print(f" - {k}: reduced -> {s.shape}, original -> {shp}")

print("\nExcluded arrays (key, original_shape, reason) - showing up to 50:")
for k, info1, info2 in excluded[:50]:
    print(f" - {k}: {info1} ({info2})")

# Plot overlays (normalized). Put `amplifier_data` on top for visibility.
if not candidates:
    print("No matching arrays found with the same sample length as 'amplifier_data'.")
else:
    # sort so amplifier_data is plotted last (on top)
    candidates.sort(key=lambda x: (x[0] == 'amplifier_data', x[0]))
    plt.figure(figsize=(12, 6))
    for k, s, _ in candidates:
        s_norm = (s - np.mean(s)) / (np.std(s) + 1e-12)
        plt.plot(t, s_norm, label=f"{k} (norm)", linewidth=0.8, alpha=0.9)
    plt.xlabel("Time (s)")
    plt.title("Overlay of arrays with an axis length equal to amplifier sample length (normalized)")
    plt.legend(loc="upper right", fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()

In [ ]:
# Print everything in amplifier_channels
amp_channels = data.get("amplifier_channels")

if amp_channels is None:
    print("No 'amplifier_channels' key found.")
elif len(amp_channels) == 0:
    print("'amplifier_channels' exists but is empty.")
else:
    print(f"amplifier_channels: {len(amp_channels)} entries")
    for i, ch in enumerate(amp_channels, start=1):
        print(f"\n--- Channel {i} ---")
        if isinstance(ch, dict):
            for key, value in ch.items():
                print(f"{key}: {value}")
        else:
            print(type(ch).__name__, ch)

In [ ]:
# Print spike_triggers
spike_triggers = data.get("spike_triggers")
if spike_triggers is None:
    print("No 'spike_triggers' found.")
elif len(spike_triggers) == 0:
    print("'spike_triggers' exists but is empty.")
else:
    print(f"spike_triggers: {len(spike_triggers)} entries")
    for i, entry in enumerate(spike_triggers, start=1):
        print(f"\n--- Entry {i} ---")
        if isinstance(entry, dict):
            for key, value in entry.items():
                print(f"{key}: {value}")
        else:
            print(type(entry).__name__, entry)

In [ ]:
# Print aux_input_channels
aux_input_channels = data.get("aux_input_channels")
if aux_input_channels is None:
    print("\nNo 'aux_input_channels' found.")
elif len(aux_input_channels) == 0:
    print("\n'aux_input_channels' exists but is empty.")
else:
    print(f"\naux_input_channels: {len(aux_input_channels)} entries")
    for i, entry in enumerate(aux_input_channels, start=1):
        print(f"\n--- Entry {i} ---")
        if isinstance(entry, dict):
            for key, value in entry.items():
                print(f"{key}: {value}")
        else:
            print(type(entry).__name__, entry)

## Ground Truth Label Reliability

> Important: In this dataset, **Dorsi-Plantar** recordings are the only condition with reliable continuous ground-truth labels for motion state (from `time_angle_allframes.mat`, where column 1 is ankle angle over time).

**What this means for labeling:**
- `DORSI_PLANTAR` folders: reliable time-aligned target signal (angle), usable for direction/neutral labeling.
- `DORSIFLEXION`, `PLANTARFLEXION`, and `PRICKING` folders: no equivalent continuous per-timepoint ground-truth target is provided in the raw files.

**Practical implication:**
- Use Dorsi-Plantar as the main supervised source when training models that require trustworthy time-varying targets.
- Treat non-Dorsi-Plantar conditions as class-level labels or proxy/derived labels unless additional annotation is added.